In [1]:
import ee
import geemap
ee.Authenticate()
ee.Initialize()

In [2]:
Map = geemap.Map()

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")
panama_fc = countries.filter(ee.Filter.eq("ADM0_NAME", "Panama"))
panama_geom = panama_fc.geometry()

Map.centerObject(panama_geom, 7)

In [3]:
districts = ee.FeatureCollection("projects/deforestation-panama/assets/Districts")

In [4]:
pa = ee.FeatureCollection("projects/deforestation-495419/assets/panama_protected_areas_polygons").filterBounds(panama_geom)

In [ ]:
#Map.addLayer(districts, {}, 'Districts')
#Map.addLayer(pa, {}, 'Panama Protected Areas Polygons')
#Map

In [6]:
print(pa.first().propertyNames().getInfo())

['WDPAID', 'REP_AREA', 'OWN_TYPE', 'REP_M_AREA', 'INT_CRIT', 'GOV_TYPE', 'ISO3', 'GIS_AREA', 'DESIG', 'STATUS', 'NO_TAKE', 'PA_DEF', 'WDPA_PID', 'CONS_OBJ', 'MARINE', 'GIS_M_AREA', 'ORIG_NAME', 'DESIG_ENG', 'MANG_PLAN', 'SUPP_INFO', 'NAME', 'DESIG_TYPE', 'METADATAID', 'MANG_AUTH', 'STATUS_YR', 'PARENT_ISO', 'NO_TK_AREA', 'VERIF', 'IUCN_CAT', 'SUB_LOC', 'system:index']


In [7]:
# Split protected areas by district

# Start with an empty FeatureCollection
empty_fc = ee.FeatureCollection([])

# Function to split protected areas by district
def split_protected_areas(district, collection):

    district = ee.Feature(district)
    collection = ee.FeatureCollection(collection)

    district_name = ee.String(district.get('Distrito'))

    # Find protected areas intersecting this district
    matching_pas = pa.filterBounds(district.geometry())

    # Intersect each protected area with the district
    def create_piece(protected_area):

        protected_area = ee.Feature(protected_area)

        pa_name = ee.String(protected_area.get('NAME'))

        # Calculate intersection geometry
        intersection = protected_area.geometry().intersection(
            district.geometry(),
            ee.ErrorMargin(1)
        )

        # Calculate areas in square kilometres
        pa_area = intersection.area(1).divide(1e6)

        district_area = district.geometry().area(1).divide(1e6)

        # Create a new feature representing the protected area piece
        return ee.Feature(intersection, {
            'PROTECTED_AREA_DISTRICT': pa_name.cat('_').cat(district_name),
            'PA_AREA_KM2': pa_area,
            'DISTRICT': district_name,
            'DISTRICT_AREA_KM2': district_area
        })

    # Create all protected-area pieces within this district
    pieces = matching_pas.map(create_piece)

    # Add pieces to the accumulating FeatureCollection
    return collection.merge(pieces)


# Iterate through all districts
protected_area_pieces = ee.FeatureCollection(
    districts.iterate(split_protected_areas, empty_fc)
)

# Remove intersections with zero area (e.g. boundary touches)
protected_area_pieces = protected_area_pieces.filter(
    ee.Filter.gt('PA_AREA_KM2', 0)
)

In [8]:
# Select only the required properties and remove geometries
import pandas as pd

pieces_simple = protected_area_pieces.map(
    lambda f: ee.Feature(None, {
        'PROTECTED_AREA_DISTRICT': f.get('PROTECTED_AREA_DISTRICT'),
        'PA_AREA_KM2': f.get('PA_AREA_KM2'),
        'DISTRICT': f.get('DISTRICT'),
        'DISTRICT_AREA_KM2': f.get('DISTRICT_AREA_KM2')
    })
)

# Retrieve features
features_info = pieces_simple.getInfo()['features']

rows = []

for f in features_info:

    props = f['properties']

    row = {
        'PROTECTED_AREA_DISTRICT': props.get('PROTECTED_AREA_DISTRICT'),
        'PA_AREA_KM2': props.get('PA_AREA_KM2'),
        'DISTRICT': props.get('DISTRICT'),
        'DISTRICT_AREA_KM2': props.get('DISTRICT_AREA_KM2')
    }

    rows.append(row)

# Create Pandas DataFrame
df = pd.DataFrame(rows)

# Round area values
df['PA_AREA_KM2'] = df['PA_AREA_KM2'].round(3)
df['DISTRICT_AREA_KM2'] = df['DISTRICT_AREA_KM2'].round(3)

# Sort by district and protected area
df = df.sort_values(
    by=['DISTRICT', 'PROTECTED_AREA_DISTRICT']
).reset_index(drop=True)

# Display dataframe
print(df)

                            PROTECTED_AREA_DISTRICT  PA_AREA_KM2   DISTRICT  \
0           La Playa de la Barqueta Agrícola_Alanje       36.024     Alanje   
1                              La Amistad_Almirante     1029.284  Almirante   
2                               Palo Seco_Almirante     1324.251  Almirante   
3    Reserva de la Biósfera de La Amistad_Almirante     1111.342  Almirante   
4                        San San Pond Sak_Almirante       21.717  Almirante   
..                                              ...          ...        ...   
184                            Isla de Cañas_Tonosí       48.971     Tonosí   
185                               La Tronosa_Tonosí      162.451     Tonosí   
186                                   Tonosí_Tonosí       16.553     Tonosí   
187              Zona de Reserva La Marinera_Tonosí        0.236     Tonosí   
188                                  Santa Fe_Ñürün       19.080      Ñürün   

     DISTRICT_AREA_KM2  
0              449.520  
1

In [9]:
# Calculate percentage of district occupied by each protected area piece
df['PA_PERCENTAGE'] = (
    df['PA_AREA_KM2'] / df['DISTRICT_AREA_KM2'] * 100
).round(2)

print(df)

                            PROTECTED_AREA_DISTRICT  PA_AREA_KM2   DISTRICT  \
0           La Playa de la Barqueta Agrícola_Alanje       36.024     Alanje   
1                              La Amistad_Almirante     1029.284  Almirante   
2                               Palo Seco_Almirante     1324.251  Almirante   
3    Reserva de la Biósfera de La Amistad_Almirante     1111.342  Almirante   
4                        San San Pond Sak_Almirante       21.717  Almirante   
..                                              ...          ...        ...   
184                            Isla de Cañas_Tonosí       48.971     Tonosí   
185                               La Tronosa_Tonosí      162.451     Tonosí   
186                                   Tonosí_Tonosí       16.553     Tonosí   
187              Zona de Reserva La Marinera_Tonosí        0.236     Tonosí   
188                                  Santa Fe_Ñürün       19.080      Ñürün   

     DISTRICT_AREA_KM2  PA_PERCENTAGE  
0          

In [ ]:
df.to_csv(
    'Panama_Protected_Area_Pieces_by_District.csv',
    index=False
)